In [2]:
import duckdb as ddb
import glob
import os
import pandas as pd
import json
from datetime import datetime
import gc

In [3]:
def sync_json_files(source_dir, target_dir):
    # 1. Get a set of all .json files in the source directory
    # Using a set allows for efficient comparison
    source_files = {f for f in os.listdir(source_dir) if f.endswith('.json')}
    
    # 2. Iterate through files in the target directory
    for filename in os.listdir(target_dir):
        # Only process .json files
        if filename not in source_files:
            file_path = os.path.join(target_dir, filename)
            try:
                os.remove(file_path)
            except Exception as e:
                print(f"Error deleting {filename}: {e}")
                continue

In [ ]:
factsDir = r'D:\EDGAR_Data_Analytics\Data\companyfacts'
submDir = r'D:\EDGAR_Data_Analytics\Data\submissions'
sync_json_files(factsDir, submDir)

In [ ]:
conn = ddb.connect('D:\\EDGAR_Data_Analytics\\Data\\secFilingsDB.db')
conn.execute("""CREATE OR REPLACE TABLE financialData (
    cik INTEGER,
    source VARCHAR,
    financialMetric VARCHAR,
    label VARCHAR,
    description VARCHAR,
    units VARCHAR,
    financialYear INTEGER,
    financialPeriod VARCHAR,
    endDate DATE,        -- Converted from string to DATE
    accn VARCHAR,
    value DOUBLE         -- Matches pandas float64
);
CHECKPOINT;""")

def parse_json_object(rowVar) ->[]:
    try:         
        df_flat = pd.DataFrame()
        df_flat = pd.json_normalize(rowVar['facts'],  max_level=1).T
        df_flat.reset_index(inplace=True)
        df_flat.columns = ['path', 'json']
        df_flat['units'] = df_flat['json'].apply(lambda x: list(x['units'].keys()))
        df_flat['label'] = df_flat['json'].apply(lambda x: x['label'])
        df_flat['description'] = df_flat['json'].apply(lambda x: x['description'])
        df_flat = df_flat.explode('units').reset_index(drop=True)
        df_flat['source'] = df_flat['path'].apply(lambda x: x.split('.')[0])
        df_flat['financialMetric'] = df_flat['path'].apply(lambda x: x.split('.')[1])
        df_flat.drop(columns=['path'], inplace=True)
        df_flat['records'] = df_flat.apply(lambda row: row['json']['units'][row['units']], axis=1)
        df_flat.drop(columns=['json'], inplace=True)
        df_flat = df_flat.explode('records').reset_index(drop=True)
        new_cols = pd.json_normalize(df_flat['records'])
        new_cols = new_cols.reindex(columns=['fy', 'fp', 'end', 'accn', 'val'])
        df_flat.drop(columns=['records'], inplace=True)
        df_flat = df_flat.join(new_cols)
        df_flat['cik'] = rowVar['cik']
        df_flat.rename(columns = {'fy':'financialYear', 'fp':'financialPeriod', 'end':'endDate', 'val':'value'}, inplace=True)
        return df_flat.to_dict(orient='records')
    except:
        return []
count=0
# 2. Iterate until all chunks are pulled
batch_size = 200
allCiks = [str(j) for j in list(conn.execute("SELECT distinct cik FROM raw_data").fetch_df()['cik'])]
print(count, datetime.now())
for i in range(0, len(allCiks), batch_size):
    batch = '('+','.join(allCiks[i:i + batch_size])+')'
    df_chunk = conn.execute(f"SELECT * FROM raw_data where cik in {batch}").fetch_df()
    df_chunk['facts'] = df_chunk['facts'].apply(json.loads)
    df_chunk['outPutListDict'] = df_chunk.apply(parse_json_object, axis=1)
    df_chunk.drop(columns='facts', inplace=True)
    df_exploded = df_chunk.explode('outPutListDict').dropna(subset=['outPutListDict'])
    expanded_cols = pd.DataFrame(df_exploded['outPutListDict'].tolist())
    del df_exploded
    del df_chunk
    gc.collect()
    conn.execute("""
                INSERT INTO financialData 
                SELECT 
                    cik,
                    source,
                    financialMetric,
                    label,
                    description,
                    units,
                    financialYear,
                    financialPeriod,
                    CAST(endDate AS DATE) AS endDate,
                    accn,
                    value
                FROM expanded_cols;
                CHECKPOINT;
                """)
    
    del expanded_cols
    gc.collect()
    count+=1
    print(count, datetime.now())


conn.execute('DROP TABLE raw_data; CHECKPOINT;')
conn.close()

0 2026-03-31 23:13:49.685410


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1 2026-03-31 23:18:03.176200


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

83 2026-04-01 02:19:25.339854


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

84 2026-04-01 02:19:51.576450
